In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd


BASE_DIR = Path(os.getcwd()).parent
load_dotenv(BASE_DIR / ".env", override=True)

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

credentials = service_account.Credentials.from_service_account_file(
    BASE_DIR / "credentials" / "service-account.json"
)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f"Conectado a {PROJECT_ID}.{DATASET_ID}")

Conectado a tc-sql-jointech.jointech


In [ ]:
# Función ejecución SQL

def run_query(query):
    result = client.query(query).to_dataframe()
    return result

In [ ]:
# Primara query: Primera visualización de datos

query = f"""
SELECT
    o.order_id,
    c.first_name,
    c.last_name,
    o.ordered_at,    
    o.order_status
FROM `{PROJECT_ID}.{DATASET_ID}.orders` AS o
JOIN `{PROJECT_ID}.{DATASET_ID}.customers` AS c
ON o.customer_id = c.customer_id
ORDER BY o.ordered_at DESC
"""

df_result = run_query(query)
df_result.head(15)

c:\Users\maxim\OneDrive\Documentos\GitHub\TEAMCHALLENGE\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,order_id,first_name,last_name,ordered_at,order_status
0,1075,Lucas,Lozano,2026-05-07 12:29:03,delivered
1,1162,Isidora,Esteve,2026-05-07 10:43:07,delivered
2,1217,Cristian,Novoa,2026-05-07 04:25:17,shipped
3,1262,Aroa,Jove,2026-05-06 19:17:31,delivered
4,1780,Bienvenida,Miguel,2026-05-06 17:17:10,returned
5,1942,Calixta,Barberá,2026-05-06 15:55:57,shipped
6,319,Paloma,Chacón,2026-05-06 10:34:36,shipped
7,336,Sandalio,Figueroa,2026-05-06 10:20:23,delivered
8,868,Plinio,Amor,2026-05-06 09:17:22,shipped
9,1073,Epifanio,Mariscal,2026-05-06 08:16:05,delivered


In [ ]:
# Segunda query: 5 productos más caros

query = f"""
SELECT 
    name, 
    sale_price 
FROM `{PROJECT_ID}.{DATASET_ID}.products`
ORDER BY sale_price DESC 
LIMIT 5
"""

df_result = run_query(query)
df_result


c:\Users\maxim\OneDrive\Documentos\GitHub\TEAMCHALLENGE\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,name,sale_price
0,Producto Smartphones Pública,1139.34
1,Producto Audio Mí,1081.14
2,Producto Wearables Queda,1078.03
3,Producto Accesorios Las,1034.59
4,Producto Audio Mayor,1026.64


In [28]:
# Tercera query: Ingresos totales

query = f"""
SELECT
    SUM(amount) AS ingresos_totales 
FROM `{PROJECT_ID}.{DATASET_ID}.payments` 
WHERE payment_status = 'completed'
"""

df_result = run_query(query)
df_result

c:\Users\maxim\OneDrive\Documentos\GitHub\TEAMCHALLENGE\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ingresos_totales
0,3865473.42


In [33]:
# Cuarta query: Categoría con más productos

query = f"""
SELECT 
    c.category_id,
    c.name AS categoria,
    COUNT(p.product_id) AS numero_productos 
FROM `{PROJECT_ID}.{DATASET_ID}.products` AS p
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.categories` AS c 
    ON p.category_id = c.category_id
GROUP BY c.category_id, c.name
ORDER BY numero_productos DESC
"""

df_result = run_query(query)
df_result

c:\Users\maxim\OneDrive\Documentos\GitHub\TEAMCHALLENGE\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_id,categoria,numero_productos
0,4,Wearables,13
1,1,Smartphones,12
2,7,Accesorios,10
3,6,Gaming,10
4,2,Laptops,9
5,3,Audio,8
6,5,Periféricos,8


In [ ]:
# Quinta query: Join de clientes y pedidos

query = f"""
SELECT
    c.first_name,
    c.last_name,
    o.ordered_at,
    COUNT(o.order_id) AS total_pedidos
FROM `{PROJECT_ID}.{DATASET_ID}.customers` AS c
JOIN `{PROJECT_ID}.{DATASET_ID}.orders` AS o
    ON c.customer_id = o.customer_id
GROUP BY c.first_name, c.last_name, o.ordered_at
ORDER BY o.ordered_at ASC
"""

df_result = run_query(query)
df_result

c:\Users\maxim\OneDrive\Documentos\GitHub\TEAMCHALLENGE\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,first_name,last_name,ordered_at,total_pedidos
0,Bernabé,Polo,2024-05-17 12:13:11,1
1,Juan Pablo,Guerra,2024-05-24 20:19:21,1
2,Maricela,Aguilera,2024-05-25 06:12:01,1
3,Encarnita,Lumbreras,2024-05-25 19:55:21,1
4,Elpidio,Barrios,2024-05-27 22:30:21,1
...,...,...,...,...
1995,Bienvenida,Miguel,2026-05-06 17:17:10,1
1996,Aroa,Jove,2026-05-06 19:17:31,1
1997,Cristian,Novoa,2026-05-07 04:25:17,1
1998,Isidora,Esteve,2026-05-07 10:43:07,1


In [ ]:
# Sexta query: Consultamos correos electrónicos de los clientes

query = f"""
SELECT
    first_name,
    last_name,
    email
FROM `{PROJECT_ID}.{DATASET_ID}.customers`
WHERE email LIKE '%@%'
ORDER BY last_name ASC
"""

df_result = run_query(query)
df_result

c:\Users\maxim\OneDrive\Documentos\GitHub\TEAMCHALLENGE\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,first_name,last_name,email
0,Jonatan,Abascal,penasdiego@example.org
1,Zacarías,Abril,asuncioncolomer@example.net
2,Lina,Acevedo,qhernandez@example.org
3,Alfonso,Aguado,ericcastro@example.com
4,Calista,Aguado,villalongavidal@example.com
...,...,...,...
495,Celia,Yáñez,casemiro53@example.com
496,Eulalia,Zabaleta,tomascristina@example.net
497,Maxi,Zapata,zestevez@example.com
498,María Manuela,Álamo,ignacia29@example.org
